# Fonctions communes : base SQLite et prétraitement

Ce notebook contient les fonctions de lecture, de nettoyage et de dédoublonnage utilisées dans les notebooks 02 et 03. Son exécution définit seulement les fonctions et les chemins : aucune base n’est créée, modifiée ou chargée.

Les cellules de code portent le tag `database-definitions` pour permettre leur chargement automatique depuis les autres notebooks et les tests.

## 1. Imports, chemins et seuil de longueur

Le seuil de 3 porte sur le nombre de caractères du texte nettoyé. Le chemin par défaut est toujours la base du sous-projet.

In [ ]:
from pathlib import Path
from contextlib import closing
import sqlite3
import re

import pandas as pd

# Trouver le sous-projet sans utiliser __file__, absent des notebooks.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le projet Bluesky ou son dossier notebooks.")
DATABASE_PATH = ROOT / "posts.db"
MIN_TEXT_LENGTH = 3


## 2. Lecture de la base

`load_posts` ouvre une base existante en lecture seule. Les publications sont triées par date décroissante, puis par URI ; leurs dates sont converties en UTC. `limit=None` charge toutes les publications. La connexion est fermée après lecture.

In [ ]:
def load_posts(database_path=DATABASE_PATH, limit=None):
    """Read without creating a database; order dates by instant, including offsets."""
    path = Path(database_path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Base de données introuvable : {path}")
    if limit is not None and limit < 1:
        raise ValueError("La limite doit être strictement positive.")
    query = """SELECT uri, did, created_at, text, langs, is_reply, has_link
               FROM posts ORDER BY julianday(created_at) DESC, uri"""
    params = ()
    if limit is not None:
        query += " LIMIT ?"
        params = (limit,)
    with closing(sqlite3.connect(path.as_uri() + "?mode=ro", uri=True)) as connection:
        posts = pd.read_sql_query(query, connection, params=params)
    posts["created_at"] = pd.to_datetime(posts["created_at"], utc=True, errors="coerce", format="mixed")
    return posts

## 3. Nettoyage d’un texte

`clean_text` retire les URL commençant par `http://`, `https://` ou `www.`, puis normalise les espaces, tabulations et retours à la ligne. La casse, les accents et la ponctuation sont conservés.

In [ ]:
def clean_text(text):
    """Remove technical noise while preserving the words of the post."""
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

## 4. Traitement complet en mémoire

`preprocess_posts` conserve les publications déclarant `fr`, ajoute une colonne `clean_text`, retire les textes trop courts et supprime les doublons exacts de texte nettoyé. Il conserve la première occurrence chargée. Le tableau d’entrée et la base SQLite restent inchangés.

Le notebook 03 détaille ces transformations et compare son résultat à cette fonction.

In [ ]:
def preprocess_posts(posts):
    """Apply language, cleaning, duplicate and content filters."""
    is_french = posts["langs"].fillna("").str.split(",").apply(lambda langs: "fr" in langs).astype(bool)
    posts = posts.loc[is_french].copy()
    posts["clean_text"] = posts["text"].map(clean_text)
    posts = posts[posts["clean_text"].str.len() >= MIN_TEXT_LENGTH]
    posts = posts.drop_duplicates(subset="clean_text", keep="first")
    return posts.reset_index(drop=True)